# Givens & Extensions



## The problem implicits solve

Some pieces of context need to be available across many calls in a chain but they are not the *subject* of any single call. The classic examples:

- An **`ExecutionContext`** for every `Future` operation in your code.
- An **`Ordering`** that tells `List.sorted` how to compare elements of a generic type.
- A **`Numeric`** instance that tells a generic `sum` method how to add elements together.
- A **logging context** carrying a trace id and a request id through every layer.
- A **Spark `Encoder`** describing how to serialize values into Spark's columnar memory.

If you thread these through manually, every method gains an extra parameter, every call site gets noisier, and you stop being able to compose generic operations cleanly. *Context parameters* let the compiler pass them for you — you declare what you need, and the compiler finds a value of that type in scope and passes it on your behalf.

## `using` parameters — declaring "I need this in context"

A `using` parameter list is a regular parameter list with one keyword change: the compiler is responsible for finding the argument, not the caller.

```scala
def sortedReverse[A](xs: List[A])(using ord: Ordering[A]): List[A] =
  xs.sorted(using ord).reverse
```

The shape:

- The method declares a second parameter list with the `using` keyword.
- At the call site, you just call `sortedReverse(xs)` — no second list. The compiler looks for an `Ordering[A]` in scope and inserts it.
- If the compiler cannot find one, you get a compile-time error: *"no given instance of type Ordering[A] was found."*

`using` parameters can be anonymous (no name) if you do not need to reference them by name in the body — most of the time you do not:

In [ ]:
// Anonymous `using` — body uses `summon` if needed, or relies on what the
// method itself calls passing the context along.
def maxOf[A](a: A, b: A)(using Ordering[A]): A =
  if summon[Ordering[A]].compare(a, b) >= 0 then a else b

println(maxOf(10, 20))           // 20, using the built-in Ordering[Int]
println(maxOf("alice", "bob"))   // bob, using Ordering[String]

// Named `using` — useful when the body needs to call methods on it directly
def joinSorted[A](xs: List[A])(using ord: Ordering[A]): String =
  xs.sorted(using ord).mkString(" < ")

println(joinSorted(List(3, 1, 4, 1, 5)))
println(joinSorted(List("zeta", "alpha", "beta")))

## `given` instances — providing a value to context

A `given` declares an instance the compiler can pick up when looking for a `using` argument. The shape:

```scala
given Ordering[Person] with
  def compare(a: Person, b: Person): Int = a.age - b.age
```

Or shorter, when the given is just a value:

```scala
given customExecutionContext: ExecutionContext = ExecutionContext.global
```

The optional name (`customExecutionContext` above) is purely for documentation and for letting you `import` or `summon` it explicitly. The given is found by *type*, not by name.

In [ ]:
case class Person(name: String, age: Int)

// A given Ordering[Person] in scope — compiler will use it when needed
given Ordering[Person] with
  def compare(a: Person, b: Person): Int = a.age - b.age

val people = List(
  Person("ganesh", 30),
  Person("alice", 25),
  Person("bob", 42),
)

// sorted picks up the given automatically — no second argument needed
println(people.sorted.map(_.name))   // List(alice, ganesh, bob)

// Multiple givens for the same type are an error — but you can scope them
object byName:
  given Ordering[Person] = Ordering.by(_.name)

import byName.given
println(people.sorted.map(_.name))   // now uses byName.given

## Where the compiler looks for `given`s — implicit scope

The compiler resolves `using` arguments by searching specific places in a fixed order. The full rules are intricate, but the practical version covers ninety-nine percent of cases:

1. **Local definitions** — `given`s defined in the current scope or any enclosing scope, including imports made into scope via `import x.given`.
2. **Companion objects of the type** — if you're looking for `Ordering[Person]`, the compiler checks the companion of `Ordering` AND the companion of `Person`.
3. **Companion objects of related types** — base types, type-argument types, etc.

The takeaway: **put your `given` instances in the companion object of one of the types involved.** They will be found automatically without anyone needing to `import` them, and there is no risk of the wrong instance being picked up from somewhere unrelated.

In [ ]:
case class Money(cents: Long)

// Define the given in Money's companion — it's always in scope when
// callers work with Money.
object Money:
  given Ordering[Money] = Ordering.by(_.cents)

val prices = List(Money(500), Money(125), Money(999), Money(250))

// No import needed — the given is found via the companion
println(prices.sorted.map(_.cents))   // List(125, 250, 500, 999)

## Type classes — the dominant pattern this enables

A **type class** is a contract that says "any type `A` that has an instance of `MyClass[A]` available can do these operations." Instead of subtyping (the type itself implements an interface), the *capability* is provided externally, by a separate value (the instance).

Three pieces define a type class:

1. **The trait** — `trait Show[A]: def show(a: A): String`. The contract.
2. **Given instances** — for the types that should support the operation. `given Show[Int]: def show(n: Int) = n.toString`.
3. **A summoner method** — usually a `using` parameter on a generic method, possibly with an extension method for sugar at the call site.

The win over subtyping:

- **You can add capabilities to types you do not own.** `String` and `Int` are in the standard library, but you can write a `Show` instance for both.
- **You can have multiple instances** scoped differently. `Person` can be `Ordered` by age in one module, by name in another.
- **The capability is decoupled from the type.** A `Person` is not "a Show" — it just *has* a `Show` instance available.

In [ ]:
// 1. The trait — the contract
trait Show[A]:
  def show(a: A): String

// 2. Given instances — for the types you want to support
given Show[Int] with
  def show(n: Int): String = s"int($n)"

given Show[String] with
  def show(s: String): String = s"string(\"$s\")"

// A given that depends on another given — Show[List[A]] needs Show[A]
given showList[A](using inner: Show[A]): Show[List[A]] with
  def show(xs: List[A]): String =
    xs.map(inner.show).mkString("[", ", ", "]")

// 3. A summoner method using the type class
def describe[A](a: A)(using s: Show[A]): String = s.show(a)

println(describe(42))                       // int(42)
println(describe("hello"))                  // string("hello")
println(describe(List(1, 2, 3)))            // [int(1), int(2), int(3)]
println(describe(List("a", "b")))           // [string("a"), string("b")]
println(describe(List(List(1), List(2,3)))) // [[int(1)], [int(2), int(3)]]

## `summon` and explicit context lookup

`summon[T]` returns the given instance of type `T` from scope. Useful inside a method body when you want to use the given instance directly without naming it as a `using` parameter.

It is essentially shorthand for `the[T]` from Scala 2 days, and it composes nicely with type-class designs.

In [ ]:
trait Show[A]:
  def show(a: A): String

given Show[Int] with
  def show(n: Int): String = s"int($n)"

def showIt[A: Show](a: A): String =
  // [A: Show] is a "context bound" — sugar for (using Show[A])
  // summon[Show[A]] retrieves the instance to use
  summon[Show[A]].show(a)

println(showIt(42))    // int(42)

## Extension methods — adding methods to types you don't own

An `extension` block adds methods to an existing type. The methods look like they're defined on the type, but they're really just regular methods translated by the compiler. No subclassing, no monkey-patching, no runtime cost beyond the method call itself.

The shape:

```scala
extension (s: String)
  def shout: String = s.toUpperCase + "!"
  def squareIt: String = s + s

"hello".shout      // "HELLO!"
"abc".squareIt     // "abcabc"
```

This is how you'd add `.toJson` to your own case classes, `.snakeCase` to `String`, or any domain-specific shorthand.

In [ ]:
extension (s: String)
  def shout: String = s.toUpperCase + "!"
  def reverseWords: String = s.split(" ").reverse.mkString(" ")
  def truncate(n: Int): String =
    if s.length <= n then s else s.take(n) + "..."

println("hello".shout)                            // HELLO!
println("the quick brown fox".reverseWords)       // fox brown quick the
println("the long sentence here".truncate(10))    // the long s...

// Extension on a generic type
extension [A](xs: List[A])
  def second: A          = xs(1)
  def safeHead: Option[A] = if xs.isEmpty then None else Some(xs.head)

println(List(10, 20, 30).second)        // 20
println(List.empty[Int].safeHead)       // None
println(List(1).safeHead)               // Some(1)

## Extensions + type classes — the receiver sugar

Combine an extension method with a type-class lookup, and you get the "calls a method on the value, but the implementation comes from a given instance" pattern. This is exactly how the standard library's `xs.sorted` works.

```scala
trait Show[A]:
  def show(a: A): String

extension [A](a: A)(using s: Show[A])
  def show: String = s.show(a)

// Now anywhere a Show[A] is in scope, you can call a.show:
42.show
"hello".show
List(1, 2, 3).show
```

This style reads better than `describe(42)` and is what most modern Scala libraries adopt. It is what Cats's `MyValue.asJson` does, what ZIO's `effect.flatMap` does, and what a whole layer of derivative ergonomics in Scala 3 rest on.

In [ ]:
trait Show[A]:
  def show(a: A): String

given Show[Int] with
  def show(n: Int) = s"int($n)"

given Show[String] with
  def show(s: String) = s"\"$s\""

given listShow[A](using s: Show[A]): Show[List[A]] with
  def show(xs: List[A]) = xs.map(s.show).mkString("[", ", ", "]")

// Extension method that uses the type-class instance for the receiver
extension [A](a: A)(using s: Show[A])
  def show: String = s.show(a)

// Call it method-style — the implementation comes from the given
println(42.show)                        // int(42)
println("hi".show)                      // "hi"
println(List(1, 2, 3).show)             // [int(1), int(2), int(3)]
println(List("a", "b").show)            // ["a", "b"]

## Context bounds — `[A: TypeClass]`

When a generic method needs an instance of a type class, the literal way to write it is:

```scala
def maxOf[A](xs: List[A])(using ord: Ordering[A]): A = ...
```

That is verbose. Scala lets you abbreviate with a **context bound**:

```scala
def maxOf[A: Ordering](xs: List[A]): A = ...
```

`[A: Ordering]` is sugar for "there is a `using Ordering[A]` parameter on this method." Inside the body, you retrieve the instance with `summon[Ordering[A]]`. The context-bound form is what you'll see in most Scala 3 library code.

In [ ]:
// Verbose form
def maxVerbose[A](xs: List[A])(using ord: Ordering[A]): A =
  xs.reduce((a, b) => if ord.gt(a, b) then a else b)

// Concise — context bound
def maxConcise[A: Ordering](xs: List[A]): A =
  val ord = summon[Ordering[A]]
  xs.reduce((a, b) => if ord.gt(a, b) then a else b)

println(maxConcise(List(3, 1, 4, 1, 5, 9, 2, 6)))
println(maxConcise(List("alpha", "zeta", "beta")))

// Multiple context bounds — composes
def show[A: Show: Ordering](xs: List[A]): String =
  xs.sorted.map(_.show).mkString(", ")

trait Show[A]:
  def show(a: A): String
given Show[Int] with
  def show(n: Int): String = s"$n"

extension [A](a: A)(using s: Show[A])
  def show: String = s.show(a)

println(show(List(3, 1, 4, 1, 5)))

## `using` chains — composing environments

A method's `using` parameter list can require *multiple* contexts at once. Each one becomes a separate lookup. This is how you compose larger environments — a method that uses both an `ExecutionContext` and a `Tracer` just lists both.

In [ ]:
trait Tracer:
  def trace(msg: String): Unit

trait Clock:
  def now: Long

// Method requiring two contexts
def loggedOp[A](name: String)(body: => A)(using tracer: Tracer, clock: Clock): A =
  val start = clock.now
  tracer.trace(s"[$name] starting at $start")
  val result = body
  tracer.trace(s"[$name] done in ${clock.now - start} ms")
  result

// Provide both givens
given Tracer with
  def trace(msg: String): Unit = println(s"TRACE: $msg")

given Clock with
  def now: Long = System.currentTimeMillis()

val result = loggedOp("compute") {
  Thread.sleep(10)
  42 * 42
}
println(s"result = $result")

## Conversions — `given Conversion[A, B]`

A `given Conversion[A, B]` tells the compiler "an `A` may be silently converted to a `B`." This is the modern, scoped form of Scala 2's free-for-all `implicit def a2b`.

Use sparingly. Automatic conversions can hide intent and produce confusing error messages. They are most defensible at *boundaries* — converting a `String` to a `Path` at I/O boundaries, converting a domain primitive to its wrapping case class.

To opt in, you must `import scala.language.implicitConversions` in any file that uses one. The friction is intentional.

In [ ]:
import scala.language.implicitConversions

case class UserId(value: String)

// A conversion in scope — any String can become a UserId where one is expected
given Conversion[String, UserId] = (s: String) => UserId(s)

def findUser(id: UserId): String = s"finding ${id.value}"

// Pass a String where a UserId is expected — conversion fires
println(findUser("u-001"))      // finding u-001
println(findUser(UserId("u-002")))  // finding u-002 — explicit also works

## Standard library and ecosystem ay-pee-eyes that use this

A short tour, so you recognize the pattern when you read other people's code.

- **`Ordering[A]`** — passed implicitly to `sorted`, `sortBy`, `min`, `max`.
- **`Numeric[A]`** — passed implicitly to generic `sum`, `product`.
- **`ExecutionContext`** — required by every `Future` combinator. We will see this in notebook 09.
- **`ClassTag[A]`** — required by `new Array[A](size)` because of erasure (notebook 07).
- **Spark `Encoder[T]`** — Spark uses a type class called `Encoder` to know how to serialize a `T` into the columnar in-memory format. `spark.implicits._` brings the basic instances into scope.
- **Circe / uPickle / Play JSON codecs** — every modern Scala JSON library uses type classes for serialization. You write `case class User(name: String, age: Int)` and get a JSON codec by importing the derivation machinery.

When you see a method signature with a `using` parameter list (or in Scala 2 code, an `implicit` parameter), the question to ask is: *what's the contract this method is asking the compiler to fulfill?*

## Scala 2 → Scala 3 quick reference

If you read a Scala 2 codebase — and you will, because Spark and most of the existing ecosystem is Scala 2 — here's the keyword translation:

| Scala 2 | Scala 3 |
|---|---|
| `implicit def add(...) (implicit x: Ec): R` | `def add(...)(using x: Ec): R` |
| `implicit val intShow: Show[Int] = ...` | `given Show[Int] = ...` (or `given Show[Int] with ...`) |
| `implicit class StringOps(s: String) { def shout = ... }` | `extension (s: String) def shout = ...` |
| `implicitly[Ordering[Int]]` | `summon[Ordering[Int]]` |
| `implicit conversion`s | `given Conversion[A, B] = ...` + `import scala.language.implicitConversions` |

The mechanics are the same; the keywords are clearer. Spark code you read will use the left column.